# Rakuten Product Image Classification

The goal of this notebook is to preprocess the product images and train a pretrained CNN (EfficientNet-B0) to predict the product category (`prdtypecode`) from the image.

# 1. Setup and Imports

`torch` and the device are initialised **before** all other imports. On Apple Silicon (MPS) a different import order can cause segfaults inside Jupyter.

In [ ]:
import os

os.chdir(os.path.expanduser('~/Desktop/Alles/Liora/Rakuten_Project'))  # change to your specific path
BASE_DIR = os.getcwd()
IMAGES_DIR = os.path.join(BASE_DIR, 'images')
DATA_DIR = os.path.join(BASE_DIR, 'text')
IMAGES_CROPPED_DIR = os.path.join(BASE_DIR, 'images', 'cropped_image_train')
IMAGES_CROPPED_DIR_TEST = os.path.join(BASE_DIR, 'images', 'cropped_image_test')

In [ ]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

In [ ]:
import pickle

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchinfo import summary
from torchvision import transforms

# 2. Image Preprocessing

Some raw images have a large white margin around the product, so they are cropped and resized before training.

The cropping works as follows: the average pixel value of the four corners is used as a reference for the background colour. A threshold of 85% of this corner average is applied — pixels darker than the threshold are considered part of the product. The image is then cropped to the bounding box of those pixels. If no dark pixels are found, or if the bounding box has no area, the image is simply resized to 224x224 without cropping. This allows dynamic cropping regardless of the background colour. 224x224 is the input size expected by pretrained models such as EfficientNet-B0.

In [ ]:
def cropping(image):
    image = image.convert('RGB')
    img_array = np.array(image)
    top_left = img_array[0, 0].mean()
    top_right = img_array[0, -1].mean()
    bottom_left = img_array[-1, 0].mean()
    bottom_right = img_array[-1, -1].mean()
    corners = (top_left + top_right + bottom_left + bottom_right) / 4
    threshold = corners * 0.85
    rows, cols, _ = np.where(img_array < threshold)
    if rows.size == 0:
        res = image.resize((224, 224))
    else:
        top, bottom = rows.min(), rows.max()
        left, right = cols.min(), cols.max()
        if top == bottom or left == right:
            res = image.resize((224, 224))
        else:
            res = image.crop((left, top, right, bottom))
            res = res.resize((224, 224))
    return res

In [ ]:
# Example: raw image before cropping
img = Image.open(IMAGES_DIR + '/image_train/image_482853_product_928819.jpg')
img

In [ ]:
# The same image after cropping
cropping(img)

## Cropping all images

A new folder holds the cropped images. The loop applies the cropping function to every image and saves it there.

Note: this only needs to run once. `exist_ok=True` is deliberately not used, so the folder is not silently reused if it already contains images from a previous run.

In [ ]:
os.makedirs('images/cropped_image_test')
file_names = os.listdir(IMAGES_DIR + '/image_train')
for i in file_names:
    img = Image.open(os.path.join('images/image_train/', i))
    cropped_img = cropping(img)
    cropped_img.save(os.path.join('images/cropped_image_train/', i))

# 3. Data Preparation

The training table and the labels are joined, and the path to the cropped image is built for every product from `imageid` and `productid` (`os.listdir()` does not preserve the DataFrame order, so the file name is the only reliable link between image and label).

In [ ]:
X_train_unprocessed = pd.read_csv(DATA_DIR + '/X_train_update.csv', index_col='Unnamed: 0')
y_train_unprocessed = pd.read_csv(DATA_DIR + '/Y_train_CVw08PX.csv', index_col='Unnamed: 0')

df_full = X_train_unprocessed.join(y_train_unprocessed)
preprocessed_df_image = df_full[['imageid', 'productid', 'prdtypecode']]
preprocessed_df_image['image_path'] = (
    IMAGES_CROPPED_DIR + '/image_' + preprocessed_df_image['imageid'].astype(str)
    + '_product_' + preprocessed_df_image['productid'].astype(str) + '.jpg'
)

In [ ]:
# Check that every row has an existing image file
preprocessed_df_image['file_exists'] = preprocessed_df_image['image_path'].apply(os.path.exists)
print(preprocessed_df_image['file_exists'].value_counts())
preprocessed_df_image.drop(columns=['file_exists'], inplace=True)

## Train / validation split

By default the full data frame is used. For faster iteration while testing the pipeline, `preprocessed_df_image.sample(n, random_state=42)` can be used instead — the fixed `random_state` keeps results comparable between experiments.

The class labels are not in the range 0-26 that PyTorch requires. `LabelEncoder` maps the original class codes (e.g. 10, 40, 2280) to consecutive integers (0, 1, ... 26). `stratify=y_image_encoded` is needed because the classes are not equally distributed.

In [ ]:
# Use the full dataset. Swap to e.g. preprocessed_df_image.sample(10000, random_state=42)
# for quicker experimentation.
data_to_use = preprocessed_df_image
X_image = data_to_use['image_path']
y_image = data_to_use['prdtypecode']

le = LabelEncoder()
y_image_encoded = le.fit_transform(y_image)

X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(
    X_image, y_image_encoded, test_size=0.25, random_state=42, stratify=y_image_encoded
)

In [ ]:
# Double check that the encoding worked
print("Min label:", y_train_img.min(), "Max label:", y_train_img.max())
print("Number of unique labels:", len(set(y_train_img)))

In [ ]:
# The encoder can be reused by the multimodal notebook, but I decided to go with the Decoder from the Text model
joblib.dump(le, 'image_label_encoder.pkl')

## Dataset class

PyTorch needs a custom Dataset class to load the images. `RakutenDataset` inherits from `Dataset` and implements three methods:

- `__init__`: stores image paths, labels and optional transforms
- `__len__`: returns the number of images
- `__getitem__`: opens a single image by index, converts it to RGB, applies the transforms and returns the image-label pair

The DataLoader uses this class to load the images in batches during training.

In [ ]:
class RakutenDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths.tolist()
        self.labels = labels.tolist()
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

## Preprocessing config

`timm` exposes the exact preprocessing configuration (mean, std, interpolation) a pretrained model was trained with. It is read once and reused everywhere instead of hardcoding the values, so the transforms always match the model.

In [ ]:
_reference_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
data_config = timm.data.resolve_data_config({}, model=_reference_model)
del _reference_model
print(data_config)

# 4. Training Function

`train_experiment` builds and trains a **fresh** EfficientNet-B0 for one experiment configuration. Each boolean flag turns a technique on or off, so combinations can be compared fairly, always starting from the same pretrained weights:

- `use_new_classifier`: replaces the final linear layer with a small `Linear -> ReLU -> Dropout -> Linear` head
- `use_freezing`: freezes all layers except `blocks.5`, `blocks.6`, `conv_head`, `bn2` and the classifier
- `use_augmentation`: random rotation, resized crop and colour jitter on the training images only
- `use_weights`: weights the loss inversely proportional to class frequency
- `use_sampler`: alternative to `use_weights` — oversamples rare classes in the DataLoader instead of reweighting the loss

Training uses early stopping on the validation loss (`patience` epochs without improvement) and separately saves the model whenever a new best F1 is reached. The two are independent, since the epoch with the lowest validation loss is not always the epoch with the best F1.

In [ ]:
def train_experiment(use_freezing=False, use_augmentation=False,
                     use_weights=False, use_new_classifier=False,
                     use_sampler=False, num_epochs=20, patience=3,
                     batches=32, model_save_path='best_model.pth'):

    # Class distribution, needed by use_weights and/or use_sampler
    _, counts_classes = np.unique(y_train_img, return_counts=True)
    total = sum(counts_classes)
    length = len(counts_classes)
    weight_per_class = [total / (length * counts_classes[i]) for i in range(length)]

    # 1. Fresh pretrained model
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
    model = model.to(device)

    # 2. Optional: new classifier head
    if use_new_classifier:
        model.classifier = nn.Sequential(
            nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
        )
        model = model.to(device)

    # 3. Optional: freeze early layers
    if use_freezing:
        for param in model.parameters():
            param.requires_grad = False
        for module in [model.blocks[5], model.blocks[6],
                       model.conv_head, model.bn2, model.classifier]:
            for param in module.parameters():
                param.requires_grad = True

    # 4. Transforms
    transform_train = transforms.Compose([
        transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomRotation(10),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=data_config['mean'], std=data_config['std'])
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=data_config['mean'], std=data_config['std'])
    ])

    train_transform_to_use = transform_train if use_augmentation else transform_val

    train_dataset = RakutenDataset(image_paths=X_train_img, labels=y_train_img,
                                   transform=train_transform_to_use)
    val_dataset = RakutenDataset(image_paths=X_val_img, labels=y_val_img,
                                 transform=transform_val)

    # 5. DataLoader, with or without weighted sampling
    if use_sampler:
        sample_weights = [weight_per_class[label] for label in y_train_img]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                        replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=batches, sampler=sampler)
    else:
        train_loader = DataLoader(train_dataset, batch_size=batches, shuffle=True)

    val_loader = DataLoader(val_dataset, batch_size=batches, shuffle=False)

    # 6. Loss function, optionally weighted by class frequency
    if use_weights:
        class_weights = torch.tensor(weight_per_class, dtype=torch.float32).to(device)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    # 7. Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    best_f1 = 0
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'f1': [], 'accuracy': []}

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_all_labels = []
        train_all_predicted = []

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_all_labels.extend(labels.cpu().numpy())
            train_all_predicted.extend(predicted.cpu().numpy())

        train_f1 = f1_score(train_all_labels, train_all_predicted, average='weighted')

        model.eval()
        val_loss = 0
        correct = 0
        total_val = 0
        all_labels = []
        all_predicted = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predicted.extend(predicted.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        avg_train_loss = train_loss / len(train_loader)
        accuracy = correct / total_val
        f1 = f1_score(all_labels, all_predicted, average='weighted')

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Train F1: {train_f1:.4f} | "
              f"Val Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f}")

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_f1'].append(train_f1)
        history['f1'].append(f1)
        history['accuracy'].append(accuracy)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early Stopping!")
                break

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), model_save_path)

    return model, best_f1, history, val_loader

# 5. Ablation Study (10k sample)

Each technique and combination is tested on a fixed 10,000-row sample for faster iteration, with a fixed `random_state` so the runs stay comparable. Everything is compared against the same baseline (no freezing, no augmentation, no weighting, original classifier).

To run this section, re-run the train/validation split above with `preprocessed_df_image.sample(10000, random_state=42)` instead of the full data frame.

In [ ]:
results_10k = {}

results_10k['baseline'] = train_experiment()[1:]
results_10k['freezing'] = train_experiment(use_freezing=True)[1:]
results_10k['augmentation'] = train_experiment(use_augmentation=True)[1:]
results_10k['weights'] = train_experiment(use_weights=True)[1:]
results_10k['sampler'] = train_experiment(use_sampler=True)[1:]
results_10k['classifier'] = train_experiment(use_new_classifier=True)[1:]

results_10k['freezing_augmentation'] = train_experiment(use_freezing=True, use_augmentation=True)[1:]
results_10k['freezing_classifier'] = train_experiment(use_freezing=True, use_new_classifier=True)[1:]
results_10k['freezing_weights'] = train_experiment(use_freezing=True, use_weights=True)[1:]
results_10k['freezing_sampler'] = train_experiment(use_freezing=True, use_sampler=True)[1:]
results_10k['freezing_classifier_weights'] = train_experiment(use_freezing=True, use_new_classifier=True, use_weights=True)[1:]
results_10k['freezing_classifier_sampler'] = train_experiment(use_freezing=True, use_new_classifier=True, use_sampler=True)[1:]
results_10k['freezing_augmentation_weights'] = train_experiment(use_freezing=True, use_augmentation=True, use_weights=True)[1:]
results_10k['freezing_augmentation_sampler'] = train_experiment(use_freezing=True, use_augmentation=True, use_sampler=True)[1:]
results_10k['freezing_augmentation_classifier'] = train_experiment(use_freezing=True, use_augmentation=True, use_new_classifier=True)[1:]
results_10k['freezing_augmentation_classifier_weights'] = train_experiment(use_freezing=True, use_augmentation=True, use_new_classifier=True, use_weights=True)[1:]
results_10k['freezing_augmentation_classifier_sampler'] = train_experiment(use_freezing=True, use_augmentation=True, use_new_classifier=True, use_sampler=True)[1:]

results_10k['augmentation_weights'] = train_experiment(use_augmentation=True, use_weights=True)[1:]
results_10k['augmentation_sampler'] = train_experiment(use_augmentation=True, use_sampler=True)[1:]
results_10k['augmentation_classifier'] = train_experiment(use_augmentation=True, use_new_classifier=True)[1:]
results_10k['augmentation_classifier_weights'] = train_experiment(use_augmentation=True, use_new_classifier=True, use_weights=True)[1:]
results_10k['augmentation_classifier_sampler'] = train_experiment(use_augmentation=True, use_new_classifier=True, use_sampler=True)[1:]

results_10k['classifier_weights'] = train_experiment(use_new_classifier=True, use_weights=True)[1:]
results_10k['classifier_sampler'] = train_experiment(use_new_classifier=True, use_sampler=True)[1:]

In [ ]:
with open('results_10k.pkl', 'wb') as f:
    pickle.dump(results_10k, f)

In [ ]:
summary_10k = pd.DataFrame({
    'experiment': list(results_10k.keys()),
    'best_f1': [values[0] for values in results_10k.values()]
})
summary_10k = summary_10k.sort_values('best_f1', ascending=False)
print(summary_10k)

In [ ]:
for name, (f1, hist, _) in results_10k.items():
    epochs_run = len(hist['train_loss'])
    print(f"{name}: Best F1 = {f1:.4f}, Epochs = {epochs_run}")

In [ ]:
best_experiment = summary_10k.loc[summary_10k['best_f1'].idxmax(), 'experiment']
colors = ['#e74c3c' if exp == best_experiment else '#3498db' for exp in summary_10k['experiment']]

plt.figure(figsize=(12, 6))
sns.barplot(x='experiment', y='best_f1', data=summary_10k, palette=colors)
sns.despine(left=True)
plt.xticks([])
plt.xlabel('Tested Combinations (Freezing / Augmentation / Weights / Sampler / Classifier)')
plt.ylabel('F1 Score (weighted)')
plt.title('Ablation Study Results (10k Sample)', pad=20)

best_position = summary_10k.reset_index(drop=True).index[summary_10k['experiment'] == best_experiment][0]
best_value = summary_10k.loc[summary_10k['experiment'] == best_experiment, 'best_f1'].values[0]
plt.text(best_position, best_value + 0.01, f'{best_value:.4f}',
         ha='center', va='bottom', fontweight='bold')

cnn_best = 0.3093  # best F1 of the SimpleCNN baseline (see section 8)
plt.axhline(y=cnn_best, color='black', linestyle='--', linewidth=1.5)

plt.show()

# 6. Final Training Run

Best configuration from the ablation study, trained on the full dataset. Re-run the train/validation split above with the full `preprocessed_df_image` (not a sample) before running this cell.

Note: on the full 77,344-row dataset epoch 4 reached the highest F1 in an earlier run (Train Loss: 0.5370, Val Loss: 1.3107, Val Accuracy: 0.6415, F1 Score: 0.6382), exceeding the ResNet50 benchmark (F1 = 0.5534).

In [ ]:
results = {}
results['augmentation_classifier'] = train_experiment(
    use_augmentation=True, use_new_classifier=True, model_save_path='final_image_model.pth'
)[1:]

In [ ]:
with open('results.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
# Values of the final run on the full dataset (hardcoded so the plot can be
# regenerated without retraining)
epochs = range(1, len(history['train_loss']) + 1)
train_loss = [1.7130, 1.2524, 1.0607, 0.9142, 0.7845, 0.6721]
val_loss = [1.2938, 1.2112, 1.1889, 1.1951, 1.2481, 1.3016]
train_f1 = [0.4731, 0.6070, 0.6641, 0.7064, 0.7466, 0.7782]
val_f1 = [0.5921, 0.6282, 0.6433, 0.6458, 0.6504, 0.6515]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs, train_f1, color='#3498db', marker='o', label='Train F1')
axes[0].plot(epochs, val_f1, color='#3498db', marker='o', linestyle='--', label='Val F1')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score over epochs', pad=20)
axes[0].legend()
benchmark = 0.5534
axes[0].axhline(y=benchmark, color='black', linestyle='--', linewidth=1.5)
axes[0].text(len(epochs) - 0.25, cnbenchmarkn_best + 0.005, f'Benchmark ResNet50: {benchmark:.4f}',
             ha='right', va='bottom', color='black', fontsize=10)
axes[0].set_ylim(0.4, 1)

axes[1].plot(epochs, train_loss, color='#e74c3c', marker='s', label='Train Loss')
axes[1].plot(epochs, val_loss, color='#e74c3c', marker='s', linestyle='--', label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss over epochs', pad=20)
axes[1].legend()

plt.suptitle('Training: final image model', y=1.02)
plt.tight_layout()
plt.show()

# 7. Evaluation

The saved best model (highest F1, not necessarily the last epoch) is reloaded and evaluated. The confusion matrix is normalised per true class (row-wise percentages) rather than showing absolute counts: the classes are imbalanced, and absolute counts would make large classes look prominent regardless of how well they are actually classified.

In [ ]:
# Rebuild the architecture exactly as used for saving, then load the best weights
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
model.classifier = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
)
model.load_state_dict(torch.load('final_image_model.pth'))
model = model.to(device)
model.eval()

summary(model, input_size=(1, 3, 224, 224), device=device)

In [ ]:
transform_val = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config['mean'], std=data_config['std'])
])

val_dataset = RakutenDataset(image_paths=X_val_img, labels=y_val_img, transform=transform_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

final_labels = []
final_predicted = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        final_labels.extend(labels.cpu().numpy())
        final_predicted.extend(predicted.cpu().numpy())

In [ ]:
cm = confusion_matrix(final_labels, final_predicted)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
class_names = [f"{code}" for code in le.classes_]

plt.figure(figsize=(18, 16))
ax = sns.heatmap(cm_normalized, fmt='.2f', cmap='Reds', vmin=0, vmax=1,
                 xticklabels=class_names, yticklabels=class_names)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=15)

plt.xlabel('Predicted', fontsize=15)
plt.ylabel('True', fontsize=15)
plt.title('Confusion Matrix (% per true class)', fontsize=20)
plt.xticks(rotation=45, ha='center', fontsize=15)
plt.yticks(rotation=0, fontsize=15)
plt.tight_layout()
plt.show()

## Error clusters

Two groups of classes account for most of the confusion: toys/figurines/games (cluster A) and books/magazines/comics (cluster B).

In [ ]:
cluster_A = [1140, 1180, 1280, 1281, 1300]
cluster_B = [10, 2280, 2403, 2705]


def submatrix(codes, ax, title):
    idx = [list(le.classes_).index(c) for c in codes]
    sub = cm_normalized[np.ix_(idx, idx)]
    sns.heatmap(sub, annot=True, fmt='.2f', cmap='Reds', vmin=0, vmax=1,
                xticklabels=codes, yticklabels=codes, ax=ax,
                annot_kws={'size': 13}, cbar=False, square=True,
                linewidths=0.5, linecolor='white')
    ax.set_title(title, pad=12, fontsize=15)
    ax.set_xlabel('Predicted', fontsize=15)
    ax.set_ylabel('True', fontsize=15)
    ax.tick_params(axis='x', labelrotation=0, labelsize=13)
    ax.tick_params(axis='y', labelrotation=0, labelsize=13)
    # frame around the block
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linewidth(1.2)
        s.set_color('#444')


fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [5, 4], 'wspace': 0.35}
)
submatrix(cluster_A, axes[0], 'Cluster A')
submatrix(cluster_B, axes[1], 'Cluster B')
fig.suptitle('Confusion within the two error clusters (image model)',
             fontsize=20, y=0.98)

plt.show()  # no tight_layout - it would override wspace again

In [ ]:
report_dict = classification_report(final_labels, final_predicted,
                                   target_names=le.classes_.astype(str),
                                   output_dict=True)
f1_per_class = pd.Series({cls: report_dict[cls]['f1-score'] for cls in le.classes_.astype(str)})
f1_per_class

In [ ]:
# Number of training examples per class (y_train_img = the labels used to fit the model)
train_counts = pd.Series(y_train_img).value_counts()
train_counts.index = le.inverse_transform(train_counts.index)  # decode back to class codes

# Align counts with the class order used in the confusion matrix
counts_per_class = train_counts.reindex(le.classes_)

# Diagonal of the (already normalized) confusion matrix = per-class recall
diagonal_values = np.diag(cm_normalized)

df_plot = pd.DataFrame({
    "class_name": class_names,
    "train_count": counts_per_class.values,
    "diagonal_value": diagonal_values,
})

plt.figure(figsize=(10, 7))
plt.scatter(df_plot["train_count"], df_plot["diagonal_value"],
            color="darkorange", s=60, edgecolor="black")

for _, row in df_plot.iterrows():
    plt.annotate(
        str(row["class_name"]),
        (row["train_count"], row["diagonal_value"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8,
    )

# class counts are heavily skewed; log scale makes the trend readable
plt.xscale("log")
plt.xlabel("Number of training examples (log scale)")
plt.ylabel("Diagonal value of confusion matrix (per-class recall)")
plt.title("Per-Class Recall vs. Training Set Size")
plt.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# 8. GradCAM

GradCAM highlights the image regions the model used for its prediction. Forward and backward hooks on `conv_head` capture the activations and their gradients; the gradients are averaged per channel and used to weight the activation maps.

In [ ]:
def show_gradcam(idx, dataset_X, dataset_y, model, target_layer):
    # 1. Prepare the image
    img_path = dataset_X.iloc[idx]
    true_label = dataset_y[idx]

    image = Image.open(img_path).convert('RGB')
    input_tensor = transform_val(image).unsqueeze(0).to(device)

    # 2. Register hooks
    activations = None
    gradients = None

    def forward_hook(module, input, output):
        nonlocal activations
        activations = output.detach()

    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0].detach()

    hook_forward = target_layer.register_forward_hook(forward_hook)
    hook_backward = target_layer.register_full_backward_hook(backward_hook)

    # 3. Forward and backward pass
    model.eval()
    output = model(input_tensor)
    predicted_class = output.argmax(dim=1).item()

    model.zero_grad()
    output[0, predicted_class].backward()

    # 4. Remove the hooks again
    hook_forward.remove()
    hook_backward.remove()

    # 5. Compute GradCAM
    pooled_gradients = gradients.mean(dim=(0, 2, 3))
    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradients[i]

    heatmap = activations.mean(dim=1).squeeze()
    heatmap = torch.relu(heatmap)
    heatmap /= heatmap.max()

    # 6. Overlay the heatmap on the original image
    heatmap_np = heatmap.cpu().numpy()
    heatmap_resized = cv2.resize(heatmap_np, (224, 224))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    original_img = np.array(image.resize((224, 224)))
    overlay = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)

    true_name = le.inverse_transform([true_label])[0]
    predicted_name = le.inverse_transform([predicted_class])[0]

    plt.imshow(overlay)
    plt.title(f"True: {true_name} | Predicted: {predicted_name}")
    plt.axis('off')
    plt.show()


def show_gradcam_for_class(original_class, n_images, dataset_X, dataset_y, model, target_layer):
    # Convert the original prdtypecode (e.g. 1280) into its encoded value
    target_encoded = le.transform([original_class])[0]

    # Find the positions where this label occurs
    matching_indices = np.where(dataset_y == target_encoded)[0]
    print(f"Found {len(matching_indices)} images of class {original_class}")

    for idx in matching_indices[:n_images]:
        show_gradcam(idx, dataset_X, dataset_y, model, target_layer)

In [ ]:
# One example per class
for classes in category_names.keys():
    show_gradcam_for_class(classes, 1, X_val_img, y_val_img, model, model.conv_head)

In [ ]:
show_gradcam_for_class(1280, 5, X_val_img, y_val_img, model, model.conv_head)
show_gradcam_for_class(10, 5, X_val_img, y_val_img, model, model.conv_head)

## GradCAM for a specific confusion pair

Instead of looking at a whole class, this selects only the images where the model made one specific mistake (true class X, predicted class Y).

In [ ]:
def show_gradcam_for_confusion_pair(true_class, predicted_class, n_images,
                                    final_labels, final_predicted,
                                    dataset_X, dataset_y, model, target_layer):
    # Convert the original prdtypecodes into their encoded values
    true_encoded = le.transform([true_class])[0]
    predicted_encoded = le.transform([predicted_class])[0]

    final_labels_arr = np.array(final_labels)
    final_predicted_arr = np.array(final_predicted)

    # Find the positions where exactly this confusion happened
    matching_indices = np.where(
        (final_labels_arr == true_encoded) & (final_predicted_arr == predicted_encoded)
    )[0]

    print(f"Found {len(matching_indices)} images where true={true_class} "
          f"but predicted={predicted_class}")

    for idx in matching_indices[:n_images]:
        show_gradcam(idx, dataset_X, dataset_y, model, target_layer)

In [ ]:
show_gradcam_for_confusion_pair(
    true_class=1281, predicted_class=50, n_images=5,
    final_labels=final_labels, final_predicted=final_predicted,
    dataset_X=X_val_img, dataset_y=y_val_img, model=model, target_layer=model.conv_head
)

# 9. CNN from Scratch (baseline)

A simple CNN trained from scratch, as a reference point for the pretrained EfficientNet-B0.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=27):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 224 -> 112
        x = self.pool(torch.relu(self.conv2(x)))  # 112 -> 56
        x = self.pool(torch.relu(self.conv3(x)))  # 56 -> 28
        x = x.view(x.size(0), -1)  # flatten
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
def train_CNN(use_augmentation=False, use_weights=False, use_sampler=False,
              num_epochs=20, patience=3, batches=32,
              model_save_path='best_model_CNN.pth'):

    # Class distribution, needed by use_weights and/or use_sampler
    _, counts_classes = np.unique(y_train_img, return_counts=True)
    total = sum(counts_classes)
    length = len(counts_classes)
    weight_per_class = [total / (length * counts_classes[i]) for i in range(length)]

    # 1. Fresh model (no pretrained weights)
    model = SimpleCNN(num_classes=27)
    model = model.to(device)

    # 2. Transforms (no timm data_config here - the model is not pretrained)
    transform_train = transforms.Compose([
        transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.RandomRotation(10),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    train_transform_to_use = transform_train if use_augmentation else transform_val

    train_dataset = RakutenDataset(image_paths=X_train_img, labels=y_train_img,
                                   transform=train_transform_to_use)
    val_dataset = RakutenDataset(image_paths=X_val_img, labels=y_val_img,
                                 transform=transform_val)

    # 3. DataLoader, with or without weighted sampling
    if use_sampler:
        sample_weights = [weight_per_class[label] for label in y_train_img]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                        replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=batches, sampler=sampler)
    else:
        train_loader = DataLoader(train_dataset, batch_size=batches, shuffle=True)

    val_loader = DataLoader(val_dataset, batch_size=batches, shuffle=False)

    # 4. Loss function, optionally weighted by class frequency
    if use_weights:
        class_weights = torch.tensor(weight_per_class, dtype=torch.float32).to(device)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # 5. Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    best_f1 = 0
    history = {'train_loss': [], 'val_loss': [], 'f1': [], 'accuracy': []}

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        correct = 0
        total_val = 0
        all_labels = []
        all_predicted = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predicted.extend(predicted.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        avg_train_loss = train_loss / len(train_loader)
        accuracy = correct / total_val
        f1 = f1_score(all_labels, all_predicted, average='weighted')

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Accuracy: {accuracy:.4f} | "
              f"F1 Score: {f1:.4f}")

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['f1'].append(f1)
        history['accuracy'].append(accuracy)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early Stopping!")
                break

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), model_save_path)

    return model, best_f1, history, val_loader

In [ ]:
results_CNN = {}

results_CNN['baseline'] = train_CNN()[1:]
results_CNN['augmentation'] = train_CNN(use_augmentation=True)[1:]
results_CNN['weights'] = train_CNN(use_weights=True)[1:]
results_CNN['sampler'] = train_CNN(use_sampler=True)[1:]

In [ ]:
cnn_results = pd.DataFrame({
    'experiment': ['Baseline', 'Augmentation', 'Weights', 'Sampler'],
    'best_f1': [0.3093, 0.3063, 0.2932, 0.2615]
})

best_experiment = cnn_results.loc[cnn_results['best_f1'].idxmax(), 'experiment']
colors = ['#e74c3c' if exp == best_experiment else '#3498db' for exp in cnn_results['experiment']]

plt.figure(figsize=(12, 6))
sns.barplot(x='experiment', y='best_f1', data=cnn_results, palette=colors)
sns.despine(left=True)
plt.xlabel('')
plt.ylabel('F1 Score (weighted)')
plt.title('SimpleCNN Ablation Results (10k Sample)', pad=20)

for i, row in cnn_results.iterrows():
    plt.text(i, row['best_f1'] + 0.01, f"{row['best_f1']:.4f}",
             ha='center', va='bottom', fontweight='bold')

plt.show()

# 10. Architecture Comparison

Why EfficientNet-B0 and not ResNet50: fewer parameters at higher ImageNet accuracy.

In [ ]:
model_comparison = pd.DataFrame({
    'model': ['ResNet50', 'EfficientNetB0'],
    'parameters_m': [25.6, 5.3],
    'top1_accuracy': [74.9, 77.1]
})

colors = ['#3498db', '#e74c3c']  # ResNet50 blue, EfficientNetB0 red (chosen architecture)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(x='model', y='parameters_m', data=model_comparison, palette=colors, ax=axes[0])
sns.despine(left=True, ax=axes[0])
axes[0].set_xlabel('')
axes[0].set_ylabel('Parameters (millions)')
axes[0].set_title('Number of Parameters', pad=20)
for i, row in model_comparison.iterrows():
    axes[0].text(i, row['parameters_m'] + 0.5, f"{row['parameters_m']:.1f}M",
                 ha='center', va='bottom', fontweight='bold')

sns.barplot(x='model', y='top1_accuracy', data=model_comparison, palette=colors, ax=axes[1])
sns.despine(left=True, ax=axes[1])
axes[1].set_xlabel('')
axes[1].set_ylabel('Top-1 Accuracy (%)')
axes[1].set_title('ImageNet Accuracy', pad=20)
axes[1].set_ylim(0, 100)
for i, row in model_comparison.iterrows():
    axes[1].text(i, row['top1_accuracy'] + 1, f"{row['top1_accuracy']:.1f}%",
                 ha='center', va='bottom', fontweight='bold')

plt.suptitle('ResNet50 vs. EfficientNetB0 (keras.io benchmarks)', y=1.02)
plt.tight_layout()
plt.show()